# Íris – Base de conhecimento e mecanismos de recuperação

Este caderno documenta a construção da base de conhecimento do Íris e os mecanismos de recuperação implementados: composição da base, segmentação, embeddings, banco vetorial, busca híbrida e reordenação.

| Seção | Conteúdo |
|---|---|
| 0 | Ambiente de execução |
| 1 | Base de conhecimento |
| 2 | Segmentação (chunking) |
| 3 | Embeddings |
| 4 | Banco vetorial |
| 5 | Busca híbrida e reordenação |
| 6 | Demonstração da recuperação |
| 7 | Formato do conjunto de avaliação |

O fluxo completo de consulta, com geração de resposta e evidência das fontes, consta no caderno `02_consulta_resposta_fonte.ipynb`. A documentação de arquitetura consta em `docs/rag/arquitetura.md` e os resultados da avaliação em `docs/rag/avaliacao.md`.

Este caderno compara os mecanismos implementados. O método adotado no protótipo é a busca vetorial com Qwen3 seguida de reordenação sobre 50 candidatos (`qwen3+rerank50`).

Requisitos de execução: ambiente Python do pacote e banco PostgreSQL com pgvector em execução (ver `README.md`).

## Conceitos

Geração aumentada por recuperação (RAG, *retrieval-augmented generation*): antes da geração da resposta, o sistema recupera de uma base os trechos relevantes para a consulta e os fornece ao modelo de linguagem, que responde com base neles e indica a fonte.

O sistema tem dois fluxos.

**Ingestão**, executada a cada alteração da base:

| Etapa | Função | Implementação |
|---|---|---|
| Curadoria | Seleção dos documentos e registro de origem | `fontes.json` |
| Extração | Conversão do documento em texto, com preservação da página | `scripts/` |
| Limpeza | Remoção de cabeçalhos, rodapés e listas de referências | `ingestao.limpar` |
| Segmentação | Divisão do documento em trechos de um único assunto | `ingestao.py` |
| Metadados | Identificação de documento, página, idioma, barreira e aviso de uso | `ingestao.py` |
| Embeddings | Representação vetorial de cada trecho | `embeddings.py` |
| Banco vetorial | Armazenamento de trechos, metadados e vetores | `banco_pg.py` |

**Consulta**, executada a cada requisição:

| Etapa | Função | Implementação |
|---|---|---|
| Recuperação | Seleção dos trechos mais similares à consulta | `consultar.py` |
| Reordenação | Reavaliação dos candidatos por cross-encoder | `consultar.reranquear` |
| Geração | Resposta restrita aos trechos, com citação por frase | `gerar.py` |

In [1]:
# 0. Ambiente
import os, platform, json, time, re
from collections import Counter
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import torch, pgvector, psycopg, sentence_transformers
import ingestao, embeddings as emb, consultar, avaliar
print("Python", platform.python_version(), "| torch", torch.__version__, "| sentence-transformers",
      sentence_transformers.__version__, "| psycopg", psycopg.__version__)
print("dispositivo:", emb.dispositivo(), "| CPUs:", os.cpu_count())
print("pasta da base:", ingestao.BASE)

iris-rag/pacote/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1104 17:23:15.300000 34203 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Python 3.11.18 | torch 2.14.1 | sentence-transformers 6.2.0 | psycopg 3.3.8
dispositivo: mps | CPUs: 15
pasta da base: iris-rag


## 1 Base de conhecimento

A base está organizada em quatro partes:

| Parte | Questão atendida |
|---|---|
| A. DUA | Princípio de desenho que sustenta a adaptação |
| B. TDAH e funções executivas | Conhecimento sobre a barreira e intervenções escolares |
| C. Normas e MEC | Obrigações e vedações aplicáveis à escola |
| D. Pesquisa | Força da evidência, incluindo neuromitos |

Critérios de inclusão: fonte primária ou oficial; acesso público; licença registrada conforme declarada no documento; vigência verificada, no caso de normas; extração conferida em duas páginas de cada documento. Versões alternativas de um mesmo documento são mantidas no catálogo e excluídas do índice.

In [2]:
cat = json.load(open(ingestao.BASE / "fontes.json", encoding="utf-8"))
docs, fora = ingestao.carregar_catalogo()
print(f"{cat['n_documentos']} documentos aceitos na curadoria; {len(docs)} entram no índice")
for f in fora:
    print("   fora do índice:", f["id"], "->", f["motivo"])
print(f"{len(cat['recusados'])} recusados e {len(cat['nao_obtidos'])} não obtidos (lista em FONTES.md)\n")
print("Por parte:", dict(Counter(d["parte"] for d in docs)))
print("Por idioma:", dict(Counter(d["idioma"] for d in docs)))
print("Por tipo de evidência:", dict(Counter(d.get("tipo_evidencia") or d.get("vigencia") or "-" for d in docs)))

40 documentos aceitos na curadoria; 38 entram no índice
   fora do índice: CAST-UDL-GUIDELINES-3-0-2024 -> quase duplicata de CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024
   fora do índice: CAST-DUA-3-0-ORGANIZADOR-PTBR-2024 -> quase duplicata de CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024
22 recusados e 19 não obtidos (lista em FONTES.md)

Por parte: {'A': 4, 'B': 12, 'C': 10, 'D': 12}
Por idioma: {'pt-BR': 18, 'en': 20}
Por tipo de evidência: {'-': 4, 'diretriz': 3, 'cartilha': 2, 'revisao_narrativa': 5, 'meta-analise': 4, 'vigente': 8, 'vigente_parcial': 2, 'revisao_sistematica': 1, 'survey': 3, 'estudo_primario': 5, 'preprint': 1}


In [3]:
# Os 38, um por linha
print(f"{'parte':5s} {'id':48s} {'idioma':6s} {'evidência/vigência':20s} licença")
for d in sorted(docs, key=lambda d: (d["parte"], d["id"])):
    ev = d.get("tipo_evidencia") or d.get("vigencia") or "-"
    print(f"{d['parte']:5s} {d['id'][:48]:48s} {d['idioma']:6s} {ev[:20]:20s} {(d['licenca'] or '')[:40]}")

parte id                                               idioma evidência/vigência   licença
A     CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024            pt-BR  -                    sem licença aberta declarada (nenhuma li
A     MEC-PNEEI-CADERNO-4-PRATICAS-INCLUSIVAS-2026     pt-BR  -                    sem licença aberta declarada
A     RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA      pt-BR  -                    CC BY 4.0 (declarada na página SciELO do
A     ZERBATO-MENDES-2018-DUA-INCLUSAO-ESCOLAR         pt-BR  -                    CC BY 4.0 (declarada na p. 1 do PDF)
B     AAP-CPG-ADHD-2019                                en     diretriz             sem licença aberta declarada (manuscrito
B     ABDA-GUIA-PROFESSORES-2012                       pt-BR  cartilha             sem licença aberta declarada ('Todos os 
B     CDC-ADHD-CLASSROOM-2026                          en     cartilha             sem licença aberta declarada (página de 
B     CORTESE-2015-META-TREINO-COGNITIVO               en

Dois critérios adicionais de curadoria:

- Estudos sobre neuromitos educacionais foram incluídos para que o sistema possa indicar a ausência de fundamento de uma prática.
- Documentos que declaram evidência limitada foram incluídos, por constituírem informação relevante para a decisão do agente.

## 2 Segmentação (chunking)

1. Limpeza: remoção de cabeçalhos e rodapés repetidos, numeração de página, créditos e sumário.
2. Corte pela estrutura: títulos, em documentos PDF; cabeçalhos, em documentos HTML; artigos, em textos legais; pontos de verificação, nas diretrizes do DUA.
3. Seções com mais de 350 palavras: divisão por parágrafo, com repetição do último parágrafo (até 60 palavras). Em textos legais, repetição do caput.
4. Seções com menos de 40 palavras: agregação à seção vizinha.
5. Descarte, com contagem por motivo: referências bibliográficas, agradecimentos, tabelas de dados e as páginas 19 a 199 do Protocolo Clínico, que contêm tabelas de medicamentos.

O texto vetorizado recebe o prefixo `Documento: <título> | Seção: <seção>`.

In [4]:
t0 = time.time()
chunks, resumo = ingestao.rodar(salvar=True, verbose=True)
print(f"(levou {time.time() - t0:.1f} s; gravou chunks.jsonl e outputs/01-chunking.txt)")
t = resumo["tamanho"]
print(f"\nTamanho em palavras: mediana {t['mediana']} | p25 {t['p25']} | p75 {t['p75']} | máx {t['max']}")
print("\nDescartados:")
for k, v in sorted(resumo["descartes"].items(), key=lambda x: -x[1]):
    print(f"   {v:4d}  {k}")

38 documentos -> 1368 chunks; 372 páginas/pedaços descartados
(levou 0.6 s; gravou chunks.jsonl e outputs/01-chunking.txt)

Tamanho em palavras: mediana 215 | p25 110 | p75 305 | máx 511

Descartados:
    181  página fora do recorte (apêndice de metodologia/tabelas de remédio)
    168  trecho de referências bibliográficas
      7  trecho de tabela de dados (células soltas: Y/N, números, siglas)
      5  trecho com menos de 10 palavras
      4  página vazia (branco/só imagem)
      3  página de sumário
      2  página de diretoria/expediente (lista de nomes)
      1  página de ficha técnica / créditos
      1  trecho de créditos (lista de nomes)


In [5]:
# Trechos por documento e por parte
por_parte = Counter(c["parte"] for c in chunks)
print("Trechos por parte:", dict(por_parte))
print("Trechos por idioma:", dict(Counter(c["idioma"] for c in chunks)))
print()
for d, x in sorted(resumo["por_doc"].items(), key=lambda kv: -kv[1]["n_chunks"]):
    print(f"{x['n_chunks']:4d}  {d}")

Trechos por parte: {'A': 210, 'B': 462, 'C': 144, 'D': 552}
Trechos por idioma: {'pt-BR': 454, 'en': 914}

 124  DALGAARD-2022-INCLUSAO-META
  96  YEGENCIK-2025-META-RCT-ESCOLA
  84  SMALEJACOBSE-2019-DIFERENCIACAO-RS
  68  MEC-PNEEI-CADERNO-4-PRATICAS-INCLUSIVAS-2026
  68  AAP-CPG-ADHD-2019
  66  CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024
  66  SWELLER-2019-CARGA-COGNITIVA
  65  NICE-NG87-ADHD-2018
  55  RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA
  55  DIAMOND-2013-FUNCOES-EXECUTIVAS
  51  AGRAWAL-2024-SIMPLIFICACAO-FIDELIDADE
  44  GAASTRA-2016-META-INTERVENCOES-SALA
  41  SIMOES-2022-NEUROMITOS-BRASIL
  36  SANTOS-ALBUQUERQUE-2019-INTERVENCOES-ESCOLARES
  34  KUSTER-2018-FONTE-DYSLEXIE
  31  CORTESE-2015-META-TREINO-COGNITIVO
  31  DIOGO-2022-ADAPTACAO-CURRICULAR-RBEE
  30  MEC-POLITICA-PEEI-2008
  29  PORTARIA-MEC-421-2026-PNEEI
  28  MS-PCDT-TDAH-2022
  28  WERY-2017-FONTE-OPENDYSLEXIC
  26  DEKKER-2012-NEUROMITOS
  25  GALLIUSSI-2020-ESPACAMENTO-DISLEXIA
  24  DECRETO-12686-2025-PNEE

In [6]:
# Um trecho por dentro: o ponto de verificação do DUA sobre funções executivas
c = next(c for c in chunks if c["doc_id"] == "CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024" and "verificação" in c["secao"].lower())
for k in ["chunk_id", "doc_id", "parte", "idioma", "secao", "pagina_inicio", "barreiras", "nao_usar_para", "url", "licenca"]:
    print(f"{k:14s} {c[k]}")
print("\ntexto_indexado (o que vira vetor):\n" + c["texto_indexado"][:700] + " ...")

chunk_id       CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024#0010
doc_id         CAST-DUA-3-0-TEXTO-COMPLETO-PTBR-2024
parte          A
idioma         pt-BR
secao          [texto não reproduzido por restrição de licença]
pagina_inicio  None
barreiras      ['atencao_sustentada', 'organizacao_sequencial', 'sobrecarga_visual', 'velocidade_processamento', 'leitura_texto_longo']
nao_usar_para  []
url            https://udlguidelines.cast.org/static/cast-diretrizes-do-desenho-universal-para-a-aprendizagem.docx
licenca        sem licença aberta declarada (nenhuma licença no .docx; material CAST © CAST, Inc.)

texto_indexado (o que vira vetor):
Documento: Diretrizes Desenho Universal para Aprendizagem. Versão 3.0 (texto completo, tradução) | Seção: [texto não reproduzido por restrição de licença]
[texto não reproduzido por restrição de licença] ...


In [7]:
# Uma lei: um artigo por trecho, com a seção
for c in [c for c in chunks if c["doc_id"] == "LEI-14254-2021-DISLEXIA-TDAH"][:3]:
    print(c["chunk_id"], "|", c["secao"], "|", c["n_palavras"], "palavras")
    print("   ", re.sub(r"\s+", " ", c["texto"])[:250], "...")

LEI-14254-2021-DISLEXIA-TDAH#0000 | L14254 | 60 palavras
    L14254 L14254 Presidência da República Secretaria-Geral Subchefia para Assuntos Jurídicos LEI Nº 14.254, DE 30 DE NOVEMBRO DE 2021 Dispõe sobre o acompanhamento integral para educandos com dislexia ou Transtorno do Deficit de Atenção com Hiperativida ...
LEI-14254-2021-DISLEXIA-TDAH#0001 | Art. 1º | 69 palavras
    Art. 1º O poder público deve desenvolver e manter programa de acompanhamento integral para educandos com dislexia, Transtorno do Deficit de Atenção com Hiperatividade (TDAH) ou outro transtorno de aprendizagem. Parágrafo único. O acompanhamento integ ...
LEI-14254-2021-DISLEXIA-TDAH#0002 | Art. 2º | 68 palavras
    Art. 2º As escolas da educação básica das redes pública e privada, com o apoio da família e dos serviços de saúde existentes, devem garantir o cuidado e a proteção ao educando com dislexia, TDAH ou outro transtorno de aprendizagem, com vistas ao seu  ...


## 3 Embeddings

| Mecanismo | Modelo | Característica |
|---|---|---|
| BM25 | Sem modelo; termos sem acento e no singular | Correspondência lexical; referência de comparação |
| MPNet | `paraphrase-multilingual-mpnet-base-v2` (768 dimensões) | Busca semântica multilíngue |
| Qwen3 | `Qwen/Qwen3-Embedding-0.6B` (1.024 dimensões), com instrução na consulta | Busca semântica multilíngue para consulta curta e documento longo |

Dois terços dos trechos estão em inglês e as consultas são formuladas em português. A correspondência lexical não relaciona termos de idiomas distintos; os modelos de embeddings multilíngues aproximam expressões equivalentes. A célula seguinte apresenta a similaridade entre uma expressão em português, sua equivalente em inglês e um texto sem relação.

In [8]:
print(open("outputs/02-indexacao.txt", encoding="utf-8").read())

RELATÓRIO DE INDEXAÇÃO - Íris: base de conhecimento no PostgreSQL com pgvector
pgvector 0.8.7 | tabela iris_trechos | índice HNSW (cosseno) em cada coluna de vetor

Trechos em chunks.jsonl: 1368 (ver outputs/01-chunking.txt)
Linhas na tabela: 1368

modelo   coluna          n   dim
mpnet    vec_mpnet     1368   768
qwen3    vec_qwen3     1368  1024

BM25 (rank_bm25): montado em memória a cada processo; não é gravado no banco.



In [9]:
# O teste do idioma: mesma ideia em português e em inglês, e um trecho sem relação
modelo = consultar._modelo("mpnet")
frases = ["tempo adicional na prova para alunos com TDAH",
          "extended time accommodation on tests for students with ADHD",
          "receita de bolo de cenoura com cobertura de chocolate"]
import numpy as np
V = np.array([modelo.vetor_pergunta(f) for f in frases])
print("cosseno PT x EN (mesma ideia):   ", round(float(V[0] @ V[1]), 3))
print("cosseno PT x bolo (sem relação): ", round(float(V[0] @ V[2]), 3))
print("palavras em comum PT x EN (BM25):", set(emb.palavras(frases[0])) & set(emb.palavras(frases[1])))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15190.67it/s]

cosseno PT x EN (mesma ideia):    0.904
cosseno PT x bolo (sem relação):  -0.014
palavras em comum PT x EN (BM25): set()


## 4 Banco vetorial

O banco é o PostgreSQL com a extensão pgvector, que acrescenta o tipo de coluna `vector` e operadores de distância.

A escolha considera a evolução prevista do Íris para aplicação web com banco relacional: trechos, metadados e vetores permanecem na mesma tabela, sob o mesmo controle de acesso e a mesma rotina de cópia de segurança.

A tabela `iris_trechos` contém uma linha por trecho e uma coluna vetorial por modelo (`vec_mpnet`, `vec_qwen3`). A coluna `barreiras` é do tipo `text[]`, o que permite o filtro `'memoria_trabalho' = ANY(barreiras)`.

A tabela possui índices HNSW para busca aproximada. No volume atual, de 1.368 trechos, o otimizador do PostgreSQL executa leitura sequencial com ordenação, equivalente à busca exata.

In [10]:
# Grava trechos + vetores no PostgreSQL (lê os vetores do cache/; só calcula o que faltar)
import banco_pg
t0 = time.time()
banco_pg.rodar(consultar.modelos_prontos() or ["mpnet"])
print(f"(levou {time.time() - t0:.0f} s)")

1368 trechos gravados em iris_trechos


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19753.55it/s]

[mpnet] vetores gravados (cache; 1.1 s no banco)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/310 [00:01<05:19,  1.03s/it]

Loading weights:  97%|█████████▋| 300/310 [00:01<00:00, 361.34it/s]

Loading weights: 100%|██████████| 310/310 [00:01<00:00, 268.82it/s]

[qwen3] vetores gravados (cache; 1.6 s no banco)
prontos: ['mpnet', 'qwen3'] | tamanho da tabela com índices: 37 MB
(levou 9 s)


In [11]:
# O banco por dentro: a tabela, os índices e uma linha
con = banco_pg.conectar()
print(con.execute("select version()").fetchone()[0][:60])
print("pgvector", con.execute("select extversion from pg_extension where extname='vector'").fetchone()[0])
print()
sql_cols = ("select column_name, data_type from information_schema.columns "
            "where table_name='iris_trechos' order by ordinal_position")
for nome, tipo in con.execute(sql_cols).fetchall():
    print(f"  {nome:16s} {tipo}")
print("\nÍndices:")
for (d,) in con.execute("select indexdef from pg_indexes where tablename='iris_trechos'").fetchall():
    print("  ", d)
print("\nTrechos:", con.execute("select count(*) from iris_trechos").fetchone()[0],
      "| com vetor MPNet:", con.execute("select count(vec_mpnet) from iris_trechos").fetchone()[0],
      "| com vetor Qwen3:", con.execute("select count(vec_qwen3) from iris_trechos").fetchone()[0])

PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) on aarch64-unknow
pgvector 0.8.7

  chunk_id         text
  doc_id           text
  parte            text
  idioma           text
  tipo_evidencia   text
  titulo           text
  orgao            text
  ano              text
  url              text
  licenca          text
  secao            text
  pagina_inicio    integer
  pagina_fim       integer
  tema             ARRAY
  barreiras        ARRAY
  nao_usar_para    ARRAY
  nota_uso         text
  tipo_corte       text
  n_palavras       integer
  texto            text
  texto_indexado   text
  vec_mpnet        USER-DEFINED
  vec_qwen3        USER-DEFINED

Índices:
   CREATE UNIQUE INDEX iris_trechos_pkey ON public.iris_trechos USING btree (chunk_id)
   CREATE INDEX iris_trechos_hnsw_mpnet ON public.iris_trechos USING hnsw (vec_mpnet vector_cosine_ops)
   CREATE INDEX iris_trechos_hnsw_qwen3 ON public.iris_trechos USING hnsw (vec_qwen3 vector_cosine_ops)
   CREATE INDEX iris_trechos_doc ON pu

In [12]:
# A busca por sentido é uma consulta SQL: '<=>' é a distância cosseno do pgvector
import numpy as np
v = np.asarray(consultar._modelo("mpnet").vetor_pergunta("dividir a tarefa em passos para quem perde o fio"), dtype=np.float32)
sql = ("select doc_id, pagina_inicio, left(secao, 50), 1 - (vec_mpnet <=> %s) as cosseno "
       "from iris_trechos order by vec_mpnet <=> %s limit 5")
for doc, pag, sec, cos in con.execute(sql, [v, v]).fetchall():
    print(f"{cos:.3f}  {doc:46s} p. {pag if pag is not None else '-':>4}  | {sec}")

0.494  RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA    p.   25  | 2.4.6 Diretriz 6 - Fornecer opções para funções ex
0.491  RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA    p.    8  | 2.3.5 Como trabalhar o DUA e as deficiências curri
0.490  MS-PCDT-TDAH-2022                              p.    9  | 7.1.1. Terapia cognitivo comportamental
0.482  INEP-CARTILHA-REDACAO-ENEM-2026-DISLEXIA       p.   20  | EM DEFESA DE UM PONTO DE VISTA.
0.481  RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA    p.   26  | 2.4.6 Diretriz 6 - Fornecer opções para funções ex


In [13]:
# Filtro: a mesma consulta, sem filtro e só nas normas (parte C); o filtro vira WHERE no SQL
p = "o aluno com TDAH tem direito a acompanhamento na escola?"
for parte in [None, "C"]:
    print(f"\nfiltro_parte={parte}:")
    for x in consultar.buscar(p, "mpnet", k=5, filtro_parte=parte):
        print(f"   {x['score']:.3f}  {x['doc']}  p. {x['pagina'] or '-'}  | {x['secao'][:50]}")


filtro_parte=None:


   0.779  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 2º
   0.763  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 5º
   0.758  SANTOS-ALBUQUERQUE-2019-INTERVENCOES-ESCOLARES  p. 11  | Treinador
   0.757  CDC-ADHD-CLASSROOM-2026  p. -  | Eligibility
   0.756  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 3º

filtro_parte=C:
   0.779  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 2º
   0.763  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 5º
   0.756  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 3º
   0.749  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 4º
   0.743  LEI-14254-2021-DISLEXIA-TDAH  p. -  | Art. 1º


## 5 Busca híbrida e reordenação

Busca híbrida: o BM25 e o modelo de embeddings produzem, cada um, uma lista de 50 trechos. Cada trecho recebe a pontuação `1 / (60 + posição)` em cada lista, e as pontuações são somadas (*reciprocal rank fusion*). A fusão utiliza a posição, e não a nota, porque as escalas das notas não são comparáveis.

Reordenação: um modelo cross-encoder (`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`) avalia cada par formado pela consulta e por um trecho candidato e atribui nota de relevância. O custo é de uma inferência por par, razão pela qual a reordenação é aplicada a um conjunto limitado de candidatos. Trechos ausentes do conjunto de candidatos não são recuperados.

As células seguintes aplicam a fusão e a reordenação sobre os 20 primeiros candidatos da busca híbrida.

In [14]:
p = "tempo adicional na prova ajuda alunos com TDAH?"
d = consultar.denso_do_hibrido()
a = [cid for cid, _ in consultar.ranking(p, "bm25", 50)]
b = [cid for cid, _ in consultar.ranking(p, d, 50)]
fundido = emb.rrf([a, b], k=60)[:8]
print(f"denso do híbrido: {d}\n")
print(f"{'trecho':52s} {'pos BM25':>8s} {'pos ' + d:>10s} {'RRF':>8s}")
for cid, s in fundido:
    pa = a.index(cid) + 1 if cid in a else "-"
    pb = b.index(cid) + 1 if cid in b else "-"
    print(f"{cid[:52]:52s} {pa:>8} {pb:>10} {s:8.4f}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/310 [00:00<01:05,  4.74it/s]

Loading weights:  85%|████████▍ | 262/310 [00:00<00:00, 866.56it/s]

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 573.23it/s]

denso do híbrido: qwen3

trecho                                               pos BM25  pos qwen3      RRF
ABDA-GUIA-PROFESSORES-2012#0001                             2          1   0.0325
ABDA-GUIA-PROFESSORES-2012#0003                             7          3   0.0308
ABDA-GUIA-PROFESSORES-2012#0004                             9          5   0.0299
SANTOS-ALBUQUERQUE-2019-INTERVENCOES-ESCOLARES#0028        10          7   0.0292
ABDA-GUIA-PROFESSORES-2012#0000                            14          4   0.0291
SANTOS-ALBUQUERQUE-2019-INTERVENCOES-ESCOLARES#0035        12         11   0.0280
MS-PCDT-TDAH-2022#0021                                      1         32   0.0273
INEP-CARTILHA-REDACAO-ENEM-2026-DISLEXIA#0001               3         50   0.0250


In [15]:
t0 = time.time()
r = consultar.buscar(p, "hibrido+rerank", k=5)
print(f"híbrida + reranker ({time.time() - t0:.0f} s, inclui carregar o modelo na 1ª vez):")
print(consultar.mostrar(r, 160))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21437.60it/s]

híbrida + reranker (27 s, inclui carregar o modelo na 1ª vez):
  1. 2.6676  ABDA-GUIA-PROFESSORES-2012  p. -  [pt-BR] | TDAH - Guia para Professores - Associação Brasileira do  [AVISO não usar para: evidencia_de_eficacia]
       https://tdah.org.br/tdah-guia-para-professores/
       "[texto não reproduzido por restrição de licença]..."
  2. 2.0102  MS-PCDT-TDAH-2022  p. 11  [pt-BR] | 7.3.2. Intervenções escolares
       https://www.gov.br/conitec/pt-br/midias/protocolos/portariaconjuntan14pcdttranstornododeficitdeatencaocomhiperatividadetdah.pdf/@@display-file/file
       "O reforço simbólico é uma estratégia reativa que fornece aos alunos um reforço imediato (por exemplo, um adesivo) para atingir uma expectativa comportamental es..."
  3. 1.9265  INEP-CARTILHA-REDACAO-ENEM-2026-DISLEXIA  p. 7-9  [pt-BR] | COM DISLEXIA
       https://download.inep.gov.br/publicacoes/institucionais/avaliacoes_e_exames_da_educacao_basica/cartilha_enem_2026_dislexia.pdf
       "o acompanhamento integral p

## 6 Demonstração da recuperação

As oito consultas desta seção ilustram o funcionamento dos mecanismos de busca. Elas não constituem avaliação: não há conjunto de referência associado. A avaliação consta em `docs/rag/avaliacao.md`.

O resultado completo de cada método é gravado em `rag/outputs/03-demonstracao.txt`.

In [16]:
t0 = time.time()
demo = consultar.demonstracao(k=5)
print(f"{len(demo)} consultas em {time.time() - t0:.0f} s -> outputs/03-demonstracao.txt\n")
for p, por_metodo in demo.items():
    print("=" * 110)
    print("CONSULTA:", p)
    for m, r in por_metodo.items():
        top = r[0]
        print(f"  {m:15s} 1º: {top['doc'][:44]:44s} p. {top['pagina'] or '-':6s} [{top['idioma']}]")

8 consultas em 2 s -> outputs/03-demonstracao.txt

CONSULTA: aluno com TDAH perde o fio em enunciados longos com vários comandos na mesma questão: como reorganizar a questão sem mudar a habilidade avaliada?
  bm25            1º: MS-PCDT-TDAH-2022                            p. 11     [pt-BR]
  mpnet           1º: SANTOS-ALBUQUERQUE-2019-INTERVENCOES-ESCOLAR p. 8-9    [pt-BR]
  qwen3           1º: ABDA-GUIA-PROFESSORES-2012                   p. -      [pt-BR]
  hibrido         1º: MS-PCDT-TDAH-2022                            p. 11-12  [pt-BR]
  hibrido+rerank  1º: MS-PCDT-TDAH-2022                            p. 11     [pt-BR]
CONSULTA: dificuldade de memória de trabalho em problema de matemática de várias etapas: que apoio dar sem resolver o problema por ele?
  bm25            1º: ZERBATO-MENDES-2018-DUA-INCLUSAO-ESCOLAR     p. 6      [pt-BR]
  mpnet           1º: SWELLER-2019-CARGA-COGNITIVA                 p. 24     [en]
  qwen3           1º: SWELLER-2019-CARGA-COGNITIVA               

In [17]:
# Quanto do top-5 vem de documento em inglês, por método: o BM25 quase não chega lá
print(f"{'método':15s} {'trechos em inglês no top-5 (das 8 consultas)':>45s}")
for m in next(iter(demo.values())):
    n = sum(x["idioma"] == "en" for r in demo.values() for x in r[m])
    tot = sum(len(r[m]) for r in demo.values())
    print(f"{m:15s} {n:>3d} de {tot}")

método           trechos em inglês no top-5 (das 8 consultas)
bm25              0 de 40
mpnet            18 de 40
qwen3            17 de 40
hibrido           0 de 40
hibrido+rerank   13 de 40


In [18]:
# O que o Agente Estrategista receberia para uma consulta (híbrida + reranker)
for x in demo[consultar.PERGUNTAS_DEMO[1]]["hibrido+rerank"]:
    print(f"{x['posicao']}. {x['doc']} | p. {x['pagina'] or '-'} | {x['tipo_evidencia']} | {x['secao'][:60]}")
    if x["nao_usar_para"]:
        print("   AVISO:", x["nota_uso"][:200])
    print("  ", re.sub(r"\s+", " ", x["trecho"])[:320], "...\n")

1. DIAMOND-2013-FUNCOES-EXECUTIVAS | p. - | revisao_narrativa | Inhibitory control supports working memory
   [texto não reproduzido por restrição de licença] ...

2. ABDA-GUIA-PROFESSORES-2012 | p. - | cartilha | TDAH - Guia para Professores - Associação Brasileira do Défi
   AVISO: Cartilha sem referências: serve para linguagem com o professor, não como evidência de eficácia.
   [texto não reproduzido por restrição de licença] ...

3. SWELLER-2019-CARGA-COGNITIVA | p. 7-8 | revisao_narrativa | Compound
   Compound Self-management effect 2012 Roodenrys et al. (2012). Managing one’s own cognitive load when evidence of split attention is present, Applied Cognitive Psychology, 26, 878–886. Cognitive load effects that are found for ill-designed instructional materials (e.g. split attention) are not found when learners are ex ...

4. RBEE-SEBASTIAN-HEREDERO-2020-DIRETRIZES-DUA | p. 26-27 |  | 2.4.6 Diretriz 6 - Fornecer opções para funções executivas
   AVISO: VERSÃO 2.0: não usar para cit

## 7 Formato do conjunto de avaliação

A célula seguinte apresenta o modelo de arquivo para conjuntos de avaliação: cada item contém a consulta e os identificadores dos trechos relevantes. O script `avaliar.py` calcula recall@5, recall@10 e MRR por método, com intervalo de confiança por bootstrap.

Os identificadores de trecho dependem da segmentação; alterações na segmentação invalidam o conjunto de avaliação.

In [19]:
print(open("gabarito_modelo.json", encoding="utf-8").read())

{
  "_instrucoes": "Modelo do gabarito de busca do Íris. O GRUPO escreve as consultas no formato do Agente Estrategista (barreira + tipo de atividade + o que preservar) ANTES de olhar o que o sistema devolve. Outra pessoa marca, para cada consulta, os chunk_id de chunks.jsonl que respondem. Copie para gabarito.json e rode: python avaliar.py gabarito.json. Marque depois de congelar o chunking (se o chunking mudar, os chunk_id mudam).",
  "versao": "0",
  "partes_validas": ["A", "B", "C", "D"],
  "barreiras_validas": ["atencao_sustentada", "memoria_trabalho", "iniciacao_tarefa", "organizacao_sequencial", "sobrecarga_visual", "controle_inibitorio", "flexibilidade_cognitiva", "velocidade_processamento", "leitura_texto_longo"],
  "itens": [
    {
      "pergunta": "",
      "autor": "",
      "barreira": "",
      "tipo_atividade": "",
      "parte_esperada": "",
      "trechos_relevantes": [],
      "marcado_por": ""
    }
  ]
}



## Síntese

- 38 documentos, em quatro partes, resultaram em 1.368 trechos com metadados de origem.
- Três mecanismos de busca, a busca híbrida e a reordenação operam sobre o PostgreSQL com pgvector.
- O método adotado no protótipo, definido pela avaliação, é a busca vetorial com Qwen3 seguida de reordenação sobre 50 candidatos.
- O fluxo completo de consulta consta no caderno `02_consulta_resposta_fonte.ipynb`.